In [230]:
import torch

### Check Acceleration Support

In [231]:
# check cuda
torch.cuda.is_available()

False

In [232]:
# check mac supports pytorch acceleration
torch.backends.mps.is_available()

True

### Basics

In [233]:
tensor0d = torch.tensor(1)                #1

print(tensor0d.shape)

tensor1d = torch.tensor([1, 2, 3])        #2

print(tensor1d.shape)

tensor2d = torch.tensor([[1, 2],
                         [3, 4]])         #3

print(tensor2d.shape)

tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])  #4

print(tensor3d.shape)

torch.Size([])
torch.Size([3])
torch.Size([2, 2])
torch.Size([2, 2, 2])


In [234]:
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)

torch.int64


In [235]:
# for python floats pytorch creats tensors with 32-bit precision by default
# as 32-bit offers sufficient precision, while consuming less memory

floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

torch.float32


In [236]:
floatvec = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
print(floatvec.dtype)

torch.float64


In [237]:
floatvec = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
changed_type = floatvec.to(torch.float32)
print(changed_type.dtype)

torch.float32


In [238]:
tensor2d = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])
tensor2d

tensor([[1, 2, 3],
        [4, 5, 6]])

In [239]:
tensor2d.shape

torch.Size([2, 3])

In [240]:
print(tensor2d.reshape(3, 2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [241]:
print(tensor2d.view(3, 2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [242]:
# transpose
print(tensor2d.T)

tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [243]:
print(tensor2d.matmul(tensor2d.T))

tensor([[14, 32],
        [32, 77]])


In [244]:
print(tensor2d @ tensor2d.T)

tensor([[14, 32],
        [32, 77]])


### Autograd System

In [245]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

print(f'{z=}')
print(f'{a=}')
print(f'{loss=}')

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

z=tensor([2.4200], grad_fn=<AddBackward0>)
a=tensor([0.9183], grad_fn=<SigmoidBackward0>)
loss=tensor(0.0852, grad_fn=<BinaryCrossEntropyBackward0>)


In [246]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [247]:
print(w1.grad)
print(b.grad)

None
None


In [248]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


Custom sigmoid and binary_cross_entropy functions

In [249]:
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def binary_cross_entropy(a, y):
    # a = prediction (sigmoid output), y = target (0 or 1)
    eps = 1e-12  # to avoid log(0)
    return -(y * math.log(a + eps) + (1 - y) * math.log(1 - a + eps))


In [250]:
y = 1.0
x1 = 1.1
w1 = 2.2
b = 0.0


z = x1 * w1 + b
a = sigmoid(z)
loss = binary_cross_entropy(a, y)

print(f'{z=}')
print(f'{a=}')
print(f'{loss=}')

z=2.4200000000000004
a=0.9183397445384054
loss=0.08518786473797667


micrograd reproduction of gradients

In [251]:
import sys
sys.path.append('../src')
from micrograd import Value

In [252]:
def sigmoid(z: Value):
    return 1 / (1 + (-z).exp())

def binary_cross_entropy(a, y):
    # a = prediction (sigmoid output), y = target (0 or 1)
    eps = 1e-12  # to avoid log(0)
    return -(y * (a + eps).log() + (1 - y) * (1 - a + eps).log())

In [253]:

y = Value(1.0)
x1 = Value(1.1)
w1 = Value(2.2)
b = Value(0.0)

z = x1 * w1 + b
a = sigmoid(z)

loss = binary_cross_entropy(a, y)

print(f'{z=}')
print(f'{a=}')
print(f'{loss=}')

z=Value(data=2.4200000000000004)
a=Value(data=0.9183397445384054)
loss=Value(data=0.08518786473797667)


In [254]:
loss.backward()
print(w1.grad)
print(b.grad)

-0.08982628100765629
-0.08166025546150571


### Implementing multilayer neural networks

In [255]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):    #1
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),        #2
            torch.nn.ReLU(),                        #3

            # 2nd hidden layer
            torch.nn.Linear(30, 20),                #4
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits                               #5

In [256]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [257]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [258]:
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [259]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [260]:
torch.manual_seed(123)
X = torch.rand([1, 50])
with torch.no_grad():
    out = model(X)
    print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [261]:
torch.manual_seed(123)
X = torch.rand([1, 50])
with torch.no_grad():
    logits = model(X)
    probs = torch.softmax(logits, dim=1)
    print(probs)

tensor([[0.3113, 0.3934, 0.2952]])


### Setting up efficient data loaders

In [262]:
X_train = torch.tensor([
    [-1.2,  3.1],
    [-0.9,  2.9],
    [-0.5,  2.6],
    [ 2.3, -1.1],
    [ 2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8,  2.8],
    [ 2.6, -1.6],
])

y_test = torch.tensor([0, 1])

In [263]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        assert len(X) == len(y), "unequal X (features) and y (labels) length"
        self.features = X
        self.labels = y

    def __getitem__(self, index):                   #1
        one_x = self.features[index]                #1
        one_y = self.labels[index]                  #1
        return one_x, one_y                         #1

    def __len__(self):                              #2
        return self.labels.shape[0]                 #2


train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [264]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,       #1
    batch_size=2,
    shuffle=True,           #2
    num_workers=0           #3
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,          #4
    num_workers=0
)

In [265]:
for idx, (x, y) in enumerate(train_loader):
    print(f'Batch {idx + 1}:', x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [266]:
train_loader = DataLoader(
    dataset=train_ds,       #1
    batch_size=2,
    shuffle=True,           #2
    num_workers=0,           #3
    drop_last=True
)
for idx, (x, y) in enumerate(train_loader):
    print(f'Batch {idx + 1}:', x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


### A typical training loop

In [267]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)     #1
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)                                                      #2

num_epochs = 3
for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()                          #3
        loss.backward()                                #4
        optimizer.step()                               #5

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

model.eval(); 
with torch.no_grad():
    outputs = model(X_train)
print('--- outputs ---')
print(outputs)

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00
--- outputs ---
tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [268]:
torch.set_printoptions(sci_mode=False)
probs = torch.softmax(outputs, dim=1)
print(probs)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


In [269]:
predictions = torch.argmax(probs, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [270]:
# rather than calculating probabilties and then getiting predictions, we can directly use logits
# larger logit index will be taken for each row
predictions = torch.argmax(outputs, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [271]:
y_train

tensor([0, 0, 0, 1, 1])

In [272]:
predictions == y_train

tensor([True, True, True, True, True])

In [273]:
# number of correct predictions
torch.sum(predictions == y_train)

tensor(5)

In [274]:
def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)
            predictions = torch.argmax(logits, dim=1)
            compare = labels == predictions
            correct += torch.sum(compare)
            total_examples += len(compare)

    return (correct / total_examples).item()

In [275]:
compute_accuracy(model, train_loader)

1.0

In [276]:
compute_accuracy(model, test_loader)

1.0

### Saving and loading models

In [277]:
torch.save(model.state_dict(), "model.pth")

In [278]:
model = NeuralNetwork(2, 2)
model.load_state_dict(torch.load("model.pth"))

<All keys matched successfully>

In [279]:
compute_accuracy(model, test_loader)

1.0